# 🎓 Fine-Tuning Socratic Science Tutor on Kaggle (2x NVIDIA T4 GPUs)

- **Dataset**: [`Susu11/socratic_idea_expansion`](https://huggingface.co/datasets/Susu11/socratic_idea_expansion)
- **Target Model Output**: [`Susu11/socratic_qwen8b`](https://huggingface.co/Susu11/socratic_qwen8b)
- **Hardware**: Dual NVIDIA T4 (2x 16GB VRAM)
- **Base Architecture**: `Qwen/Qwen2.5-7B-Instruct` (or `meta-llama/Llama-3.1-8B-Instruct`)
- **Method**: QLoRA (4-bit NF4 + FP16 compute dtype + PEFT LoRA)

> **Why Instruct instead of Base?**
> Always fine-tune an **Instruct** model for conversational steering. Instruct models already understand multi-turn dialogues and role formatting, meaning your 139 Socratic examples will swiftly steer the model to use the pedagogical `<plan>` tags and the 1-question rule without catastrophic forgetting.

## 1. Verify Dual T4 GPUs

In [ ]:
!nvidia-smi

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  [GPU {i}]: {torch.cuda.get_device_name(i)} | VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")

--- 
## 2. Install Required Dependencies

In [ ]:
%%capture
!pip install -U pip
!pip install -q transformers peft trl datasets accelerate bitsandbytes wandb huggingface_hub

--- 
## 3. Authenticate (Hugging Face & Weights & Biases)
Reads `HF_TOKEN` and `WANDB_API_KEY` directly from Kaggle Secrets.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import wandb

user_secrets = UserSecretsClient()

# 1. Hugging Face Login
try:
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token, add_to_git_credential=True)
    os.environ["HF_TOKEN"] = hf_token
    print("✅ Hugging Face login successful!")
except Exception as e:
    print(f"⚠️ Could not read HF_TOKEN from Kaggle Secrets: {e}")
    login()

# 2. Weights & Biases Login
try:
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_key)
    os.environ["WANDB_API_KEY"] = wandb_key
    os.environ["WANDB_PROJECT"] = "socratic-model-fine-tune"
    print("✅ Weights & Biases login successful!")
except Exception as e:
    print(f"ℹ️ WANDB_API_KEY not found in Kaggle Secrets ({e}). Running without W&B...")
    os.environ["WANDB_DISABLED"] = "true"

--- 
## 4. (Optional) Upload Local `socratic_dataset.jsonl` to Hugging Face Hub
If you uploaded `socratic_dataset.jsonl` to Kaggle, this will push it to **`Susu11/socratic_idea_expansion`**.

In [ ]:
import json
from datasets import Dataset

DATASET_REPO_ID = "Susu11/socratic_idea_expansion"
LOCAL_FILE = "socratic_dataset.jsonl"

# If socratic_dataset.jsonl is present locally, push to Hugging Face Hub
if os.path.exists(LOCAL_FILE):
    print(f"📂 Found local '{LOCAL_FILE}'. Reading and preparing for upload...")
    records = []
    with open(LOCAL_FILE, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    
    ds_to_push = Dataset.from_list(records)
    print(f"🚀 Pushing {len(ds_to_push)} records to Hub: https://huggingface.co/datasets/{DATASET_REPO_ID} ...")
    ds_to_push.push_to_hub(DATASET_REPO_ID, private=False)
    print("🎉 Dataset uploaded to Hugging Face successfully!")
else:
    print(f"ℹ️ Local '{LOCAL_FILE}' not found. Loading directly from Hugging Face: {DATASET_REPO_ID}")

--- 
## 5. Load Dataset from Hugging Face (`Susu11/socratic_idea_expansion`)

In [ ]:
from datasets import load_dataset

DATASET_NAME = "Susu11/socratic_idea_expansion"

try:
    print(f"🌐 Loading dataset from: {DATASET_NAME}...")
    dataset = load_dataset(DATASET_NAME, split="train")
    print(f"✅ Successfully loaded {len(dataset)} examples from Hugging Face Hub!")
except Exception as e:
    print(f"⚠️ Failed to load from Hub ({e}). Falling back to local 'socratic_dataset.jsonl'...")
    records = [json.loads(l) for l in open("socratic_dataset.jsonl") if l.strip()]
    dataset = Dataset.from_list(records)
    print(f"✅ Loaded {len(dataset)} examples from local file!")

# Preview first example
print("\n--- First Example Preview ---")
for msg in dataset[0]["messages"]:
    print(f"[{msg['role'].upper()}]:\n{msg['content']}\n")

--- 
## 6. Model, Tokenizer & QLoRA Setup (Dual T4 Optimized)

> Using `Qwen/Qwen2.5-7B-Instruct` (or `meta-llama/Llama-3.1-8B-Instruct`).
> - `bnb_4bit_compute_dtype=torch.float16` for T4 Turing architecture.
> - `device_map="auto"` distributes the model across both T4 GPUs automatically.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Use Instruct model for chat/conversation fine-tuning
BASE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"  # Alternative: "meta-llama/Llama-3.1-8B-Instruct"

# 1. Load Tokenizer & Apply Chat Template
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def apply_chat_template(batch):
    return {
        "text": [
            tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False)
            for conv in batch["messages"]
        ]
    }

train_dataset = dataset.map(apply_chat_template, batched=True)
print("Sample Formatted Text:")
print(train_dataset[0]["text"][:350] + "...")

# 2. 4-bit Quantization Config (NF4 + FP16 compute for T4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # T4 optimal
    bnb_4bit_use_double_quant=True,
)

# 3. Load Model Across 2x T4
print("\nLoading model across dual GPUs with device_map='auto'...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# 4. LoRA Adapter Configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

--- 
## 7. Train with SFTTrainer & Track with W&B

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR = "./socratic_tutor_lora"
HUB_MODEL_ID = "Susu11/socratic_qwen8b"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # Effective batch size = 2 GPUs * 2 batch * 4 steps = 16
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_strategy="epoch",
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    optim="paged_adamw_8bit",
    report_to=["wandb"] if os.getenv("WANDB_API_KEY") else ["none"],
    run_name="kaggle-dual-t4-qwen-run",
    push_to_hub=False,
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args,
)

print("🚀 Starting Training on Dual T4 GPUs...")
trainer.train()

--- 
## 8. Test Socratic Tutor Inference (Verify `<plan>` Tags)

In [ ]:
system_prompt = (
    "You are a Socratic Science Tutor for a Grade 10 student. "
    "Your goal is to guide the student to discover concepts through reasoning, NEVER by giving the final answer directly. "
    "Ask EXACTLY ONE question per turn. Keep responses to 1-3 sentences. "
    "If the student is stuck, provide a simpler analogy or break the concept into a smaller step. "
    "State your pedagogical goal inside <plan>...</plan> tags."
)

test_question = "Why is the sky blue during the day but red at sunset?"

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_question}
]

prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

generated_response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- Generated Socratic Response ---")
print(generated_response)

--- 
## 9. Push Model to Hugging Face Hub (`Susu11/socratic_qwen8b`)

In [ ]:
# 1. Save Adapter Locally
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"💾 Model and tokenizer saved locally to: {OUTPUT_DIR}")

# 2. Push LoRA Adapter to Hub
print(f"📤 Pushing LoRA adapter to Hugging Face Hub: {HUB_MODEL_ID}...")
trainer.model.push_to_hub(HUB_MODEL_ID)
tokenizer.push_to_hub(HUB_MODEL_ID)
print(f"🎉 Successfully pushed LoRA adapter to: https://huggingface.co/{HUB_MODEL_ID}")

--- 
## 10. (Optional) Merge Adapter into 16-bit Base Model & Push

In [ ]:
import gc
from peft import PeftModel

MERGE_MODEL = True  # Set to True to create and upload full standalone model

if MERGE_MODEL:
    del model
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    
    print("Reloading base model in FP16 for clean merge...")
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    
    print("Merging LoRA weights with base model...")
    merged_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
    merged_model = merged_model.merge_and_unload()
    
    merged_hub_id = f"{HUB_MODEL_ID}-merged"
    print(f"📤 Pushing full merged 16-bit model to: https://huggingface.co/{merged_hub_id}...")
    merged_model.push_to_hub(merged_hub_id)
    tokenizer.push_to_hub(merged_hub_id)
    print("🎉 Standalone merged model successfully uploaded!")
else:
    print("Skipping merge step.")